# 05 · Agentic Stage-A (W2b) — tool use & the feedback gap

ShareChat-Claude is genuinely agentic. This notebook profiles the tool use and measures how often an observably-wrong action draws no human feedback.
**Backs:** `docs/safety/w2b-stageA-agentic.md`, W2b of `docs/shared-findings.md`, feedback-gap rows of `docs/reference/wildchat-vs-sharechat.md`.
**Scripts:** `w2b_agentic_turns.compute()`, `taxonomy_compare.load_agentic()`, `build_sharechat_agentic.summarize()`, `feedback_gap.compute()`.

In [1]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd(); ROOT = ROOT if (ROOT/"src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT/"src"))
import pandas as pd
pd.set_option("display.max_rows", 120)
print("repo root:", ROOT)

repo root: /data/wang/junh/githubs/human-agent-coupling-errors


## Raw scan
Doc claim: **278 tool-using turns across 162 conversations** (6.6% of llm turns).

In [2]:
import w2b_agentic_turns as w2b
a = w2b.compute()
print(f"agentic turns {a['agentic_turns']} across {a['n_convs']} convs ({a['agentic_pct']:.1f}% of {a['n_llm']} llm turns)")
print(f"consequential headers flagged: {len(a['consequential'])}")
pd.DataFrame([{"tool": k, "calls": v} for k, v in a["tool_calls"].items()])

agentic turns 278 across 162 convs (6.6% of 4182 llm turns)
consequential headers flagged: 145


,tool,calls
0,other,998
1,web_search,381
2,mcp/api,111
3,data_analysis (JS REPL),90
4,run_code (code exec),1


## Reconstruction (Stage-A record)
Doc claim: **248 agentic turns / 1,233 tool calls** (897 read-only, 336 consequential).

In [3]:
import taxonomy_compare as tc
ag = tc.load_agentic()
print(f"{ag['agentic_turns']} agentic turns | {ag['tool_calls']} tool calls "
      f"({ag['read_only']} read-only, {ag['consequential']} consequential)")
ag["op_counts"]

248 agentic turns | 1233 tool calls (897 read-only, 336 consequential)


{'act_execute': 336, 'seek_inspect': 897}

In [4]:
import build_sharechat_agentic as bsa
s = bsa.summarize(rebuild=False)   # reads existing sharechat_agentic.jsonl, does NOT rewrite it
print(f"records {s['n']} | reasoning {s['n_reason']} | tool-using turns {s['n_tool']} "
      f"across {s['n_convs']} convs | turns w/ consequential act {s['n_conseq']}")
pd.DataFrame([{"tool": k, "calls": v} for k, v in s["tools"].items()])

records 4182 | reasoning 1001 | tool-using turns 248 across 147 convs | turns w/ consequential act 49


,tool,calls
0,web_search,261
1,send,112
2,list,101
3,data_analysis,90
4,codemcp,48
5,get,44
6,brave_web_search,36
7,insert_and_execute_cell,36
8,run,27
9,read_file,25


## The feedback gap
Doc claim: feedback base rate **17.6%**; of **16** observably-wrong (tool-error) turns with a next turn, **14** drew NO human feedback (illustrative first cut).

In [5]:
import feedback_gap as fg
f = fg.compute()
br, hl, c = f["base_rate"], f["headline"], f["cells"]
print(f"base feedback rate: {br['pct']:.1f}%  ({br['fb']}/{br['n']})")
print(f"HEADLINE: {hl['nofb']}/{hl['n_wrong']} observable errors got NO feedback = {hl['pct']:.0f}%")
pd.DataFrame([["wrong (tool error)", c["wrong_fb"], c["wrong_nofb"]],
              ["ok", c["ok_fb"], c["ok_nofb"]]],
             columns=["turn", "next-turn feedback", "no feedback"]).set_index("turn")

base feedback rate: 17.6%  (577/3271)
HEADLINE: 14/16 observable errors got NO feedback = 88%


,next-turn feedback,no feedback
turn,,
wrong (tool error),2,14
ok,575,2680
